# Literature Review Agent
### Description
- Building a multiple agentic system for help researchers to save some time in literature. 

## Agents and their work
|Agents & Tool|Work description|
|------------|---------------|
|Search_Agent| The agent search research paper to the user's topic|
|Search_Tool| The tools help the agent to search the paper using the keywords|
|Downloader| The tools used to download those searched papers|
|DB_Agent| The agent read and divide into data and save those in a vector database|
|Question_Agent| THe agent generate research questions|
|Answer_agent| The agent answer those research question|
|Synthesis_Agent| Finalize the review|

### Dependencies
```bash
pip install langgraph langchain-groq langchain-openrouter chromadb \
            pypdf arxiv duckduckgo-search streamlit python-dotenv
```



## Agent : 1 - Search agent
- This agent will get topic from users input and find keywords and start searching related 

In [47]:
# Calling LLM and setup API
import os
import langchain_openrouter
import langchain_groq
from dotenv import load_dotenv

# Run API key from .env file
load_dotenv()
GROQ_API = os.getenv("GROQ_API")
OPEN_ROUTER_API = os.getenv("OPEN_ROUTER_API")

if not GROQ_API:
    raise ValueError("API keys for GROQ must be set in the .env file.")
else:
    print("GROQ API key is set.")
if not OPEN_ROUTER_API:
    raise ValueError("API keys for OPEN_ROUTER must be set in the .env file.")
else:
    print("OPEN_ROUTER API key is set.")


GROQ API key is set.
OPEN_ROUTER API key is set.


In [64]:
from langchain_groq import ChatGroq
from langchain_openrouter import ChatOpenRouter
# Search agent
llm_search = ChatGroq(model="openai/gpt-oss-20b", 
                      api_key=GROQ_API, 
                      temperature=0, 
                      max_tokens=1000)
# Testing
response = llm_search.invoke("What is Tuberculosis?")
print(f"LLM Read for search agent with model: {llm_search.model}")
print(f"Response: {response}")



LLM Read for search agent with model: openai/gpt-oss-20b
Response: content='**Tuberculosis (TB)** is a contagious infectious disease caused by the bacterium *Mycobacterium tuberculosis*. It most commonly affects the lungs (pulmonary TB) but can involve any organ (extrapulmonary TB).\n\n---\n\n## 1. How TB Spreads\n- **Airborne transmission**: When an infected person coughs, sneezes, sings, or speaks, tiny droplets containing the bacteria are released into the air.  \n- **Inhalation**: A susceptible person inhales these droplets and the bacteria can settle in the lungs.  \n- **Latency**: Most people who inhale the bacteria do not develop active disease immediately; the infection can remain dormant (latent TB) for years.\n\n---\n\n## 2. Clinical Forms\n\n| Form | Typical Features | Common Sites |\n|------|------------------|--------------|\n| **Latent TB infection (LTBI)** | No symptoms; bacteria are present but inactive | – |\n| **Pulmonary TB** | Cough (often >3\u202fweeks), chest pain

In [68]:
def get_arxiv_pdf_url(entry):
    # entry.id looks like: http://arxiv.org/abs/2506.01923v1
    arxiv_id = entry.id.split("/abs/")[-1]
    return f"https://arxiv.org/pdf/{arxiv_id}.pdf"

In [70]:
# Search query & tool implementation
import requests
import feedparser
def generate_search_queries(topic, n=3):
    prompt = f"""You are a research assistant. Generate {n} search queries for the topic: "{topic}".
    Each query should be concise and relevant to the topic. Return the queries as a list of strings.
    Return only numbered list of queries without any additional text or explanation.
    
    Topic:{topic}
    """
    response = llm_search.invoke(prompt) 
    lines = [line.strip() for line in response.content.split("\n") if line.strip()]
    queries =[]
    for line in lines:
        if line[0].isdigit():
            q = line.split('.',1)[1].strip()
            queries.append(q)
        return queries[:n] if queries else [topic]
    
def search_arxiv(query, max_results=5, retries=3, timeout=20):
    base_url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": max_results,
        "sortBy": "relevance",
        "sortOrder": "descending"
    }
    for attempt in range(retries):
        try:
            response = requests.get(base_url, params=params, timeout=timeout)
            feed = feedparser.parse(response.text)
            results = []
            for entry in feed.entries:
                pdf_url = get_arxiv_pdf_url(entry)
                results.append({
                    "title": entry.title.replace("\n", " ").strip(),
                    "authors": [a.name for a in entry.authors],
                    "abstract": entry.summary.replace("\n", " ").strip(),
                    "pdf_url": pdf_url,
                    "published": entry.published,
                    "source": "arxiv"
                })
            return results
        except Exception as e:
            print(f"arXiv attempt {attempt+1}/{retries} failed: {e}")
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))
    print(f"arXiv search failed after {retries} attempts for '{query}'")
    return []



    



In [65]:
# Semantic scholar search implementation
def search_semantic_scholar(query, max_results=5):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    params = {
        "query": query,
        "limit": max_results,
        "fields": "title,abstract,authors,url,openAccessPdf,year"
    }
    response = requests.get(url, params=params, timeout=15)
    print("Status",response.status_code, "| Body:", response.text[:200])
    data = response.json()

    results = []
    for paper in data.get("data", []):
        results.append({
            "title": paper.get("title"),
            "authors": [a["name"] for a in paper.get("authors", [])],
            "abstract": paper.get("abstract") or "",
            "pdf_url": paper.get("openAccessPdf", {}).get("url") if paper.get("openAccessPdf") else None,
            "published": paper.get("year"),
            "source": "semantic_scholar"
        })
    return results

In [66]:
r2 = search_semantic_scholar("AI-based tuberculosis detection from chest X-ray images", max_results=5)
print(r2)

Status 429 | Body: {"message": "Too Many Requests. Please wait and try again or apply for a key for higher rate limits. https://www.semanticscholar.org/product/api#api-key-form", "code": "429"}
[]


In [73]:
# Search agent
def search_agent(topic, max_results_per_query=5,display_results=True):
    queries = generate_search_queries(topic)
    print(f"Generated queries: {queries}")
    
    all_results = []
    for q in queries:
        all_results.extend(search_arxiv(q, max_results_per_query))
        # all_results.extend(search_semantic_scholar(q, max_results_per_query))
    
    seen = set()
    unique_results = []
    for r in all_results:
        if not r.get("title"):
            continue
        key = r["title"].lower().strip()
        if key not in seen:
            seen.add(key)
            unique_results.append(r)
    print(f"Found {len(unique_results)} unique papers")
    
    if display_results and unique_results:
        print("\n Search results:")
        print("=" * 60)
        for i, paper in enumerate(unique_results, 1):
            title = paper.get("title","No Title")
            authors = paper.get("authors",["Unknows"])[:2] # First 2 authors only
            year = paper.get("Year","N/A")
            
            print(f"{i:2}. Title: {title}")
            print(f"    Authors: {', '.join(authors) if authors else 'Unknown'}")
            print(f"    Year: {year}")
            print("-" * 60)
    return unique_results



In [74]:
if __name__ == "__main__":
    user_topic = "Tuberculosis detection by AI using chest X-ray images"
    max_papers = 5
    # user_topic = input("Enter a research topic: ")
    # max_papers = int(input("Enter the maximum number of papers to retrieve per query: "))
    papers = search_agent(user_topic, max_results_per_query=max_papers, display_results=True)

Generated queries: ['AI-based tuberculosis detection from chest X-ray images']
Found 5 unique papers

 Search results:
 1. Title: Classification of Pneumonia and Tuberculosis from Chest X-rays
    Authors: M. Abubakar, I. Shah
    Year: N/A
------------------------------------------------------------
 2. Title: An Efficient Mixture of Deep and Machine Learning Models for COVID-19 and Tuberculosis Detection Using X-Ray Images in Resource Limited Settings
    Authors: Ali H. Al-Timemy, Rami N. Khushaba
    Year: N/A
------------------------------------------------------------
 3. Title: Few-Shot Learning Approach on Tuberculosis Classification Based on Chest X-Ray Images
    Authors: A. A. G. Yogi Pramana, Faiz Ihza Permana
    Year: N/A
------------------------------------------------------------
 4. Title: Reliable Tuberculosis Detection using Chest X-ray with Deep Learning, Segmentation and Visualization
    Authors: Tawsifur Rahman, Amith Khandakar
    Year: N/A
---------------------

In [75]:
print(papers[4])

{'title': 'High-resolution x-ray analysis with multilayer gratings', 'authors': ['Philippe Jonnard', 'Karine Le Guen', 'Jean-Michel André'], 'abstract': 'Periodic multilayers are nowadays widely used to perform x-ray analysis in the soft x-ray range (photon energy lower than 1 keV). However, they do not permit to obtain high-resolution spectra like natural or synthetic crystals. Thus, multilayers cannot resolve interferences between close x-ray lines. It has been shown and demonstrated experimentally that patterning a grating profile within a multilayer structure leads to a diffractive optics with improved resolving power. We illustrate the use of a Mo/B4C multilayer grating in the Fe L and C K spectral ranges, around 700 eV and 280 eV respectively. First, in the Fe L range, the improved spectral resolution enables us to distinguish the Fe Lαand Lβemissions (separated by 13 eV). In addition, using a sample made of a mix of LiF and an iron ore, we show that it is possible to easily reso

## Phase 2 : Download and analyse those founded papers
### Analysing agent

In [ ]:
# Download function
import os
import requests

def download_pdf(pdf_url, save_dir="data/papers",filename=None):
    if not pdf_url or not str(pdf_url).startswith("http"):
        print(f"Invalid URL: {pdf_url}")
        return None
    os.makedirs(save_dir, exist_ok=True)
    filename = filename or pdf_url.split("/")[-1].replace(".pdf","") + ".pdf"
    filepath = os.path.join(save_dir, filename)
    
    if os.path.exists(filepath):
        print(f"Download skipped: {filename} already exists.")
        return filepath
    try:
        response = requests.get(pdf_url, timeout=30)
        if response.status_code == 200 and response.headers.get("content-type", "").startswith("application/pdf"):
            with open(filepath, "wb") as f:
                f.write(response.content)
            print(f"Downloaded: {filename}")
            return filepath
        else:
            print(f"Failed to download {filename}. Status code: {response.status_code}, Content-Type: {response.headers.get('content-type')}")
            return None
    except Exception as e:
        print(f"Error downloading {filename}: {e}")
        return None

### Extract text from pdf

In [23]:
from pypdf import PdfReader

def extract_text(filepath):
    try:
        reader = PdfReader(filepath)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text.strip()
    except Exception as e:
        print(f"Error extracting text from {filepath}: {e}")
        return ""

In [33]:
# Relevence filter
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()
GROQ_API = os.getenv("GROQ_API")
llm_relevance = ChatGroq(model="openai/gpt-oss-20b", api_key=GROQ_API, temperature=0, max_tokens=1000)

def is_relevant(abstract, topic):
    if not abstract:
        print("Empty abstract, skipping")
        return False
    prompt = f"""Topic: {topic}
    Abstract: {abstract}
    Is this paper relavant to the topic? Answer with 'Yes' or 'No' only.
    """
    response = llm_relevance.invoke(prompt)
    print("Raw response:", repr(response.content))
    return "yes" in response.content.lower().strip()

In [34]:
# Chunk test
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_text(text)

In [35]:
import torch
print(torch.cuda.is_available())

True


In [36]:
# Embed + store in chroma
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="data/vector_db")
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")

collection = client.get_or_create_collection(name="literature_review", embedding_function=embedding_fn)

def store_chunks(chunks, paper_meta, chunk_id_prefix):
    ids = [f"{chunk_id_prefix}_{i}" for i in range(len(chunks))]
    metadatas = [{"title": paper_meta["title"], "source": paper_meta["source"]} for _ in chunks]
    collection.add(documents=chunks, ids=ids, metadatas=metadatas)
    

In [60]:
# Analysis agent
def analysis_agent(papers, topic):
    stored_count = 0
    for idx, paper in enumerate(papers):
        filepath = download_pdf(paper.get("pdf_url"), filename=f"paper_{idx}.pdf")
        if not filepath:
            print(f"✗ Skipped (no PDF): {paper['title']}")
            continue

        text = extract_text(filepath)
        if not text:
            print(f"✗ Skipped (no extractable text): {paper['title']}")
            continue

        chunks = chunk_text(text)
        store_chunks(chunks, paper, chunk_id_prefix=f"paper_{idx}")
        stored_count += 1
        print(f"✓ Stored: {paper['title']} ({len(chunks)} chunks)")

    print(f"\n✅ {stored_count} papers embedded into vector DB")

In [61]:
topic = "cross-generator generalization AI-Generated text detection"
papers = search_agent(topic, max_results_per_query=5, display_results=True)
analysis_agent(papers, topic)
# for p in papers[:5]:
    # print(p["title"], "-> abstract length:", len(p.get("abstract", "")))

Generated queries: ['cross-generator generalization AI text detection']
Found 5 unique papers

 Search results:
 1. Title: Sarang at DEFACTIFY 4.0: Detecting AI-Generated Text Using Noised Data and an Ensemble of DeBERTa Models
    Authors: Avinash Trivedi, Sangeetha Sivanesan
    Year: N/A
------------------------------------------------------------
 2. Title: Faith in AI can narrow the futures individuals consider
    Authors: Aoi Naito, Hirokazu Shirado
    Year: N/A
------------------------------------------------------------
 3. Title: Foundations of GenIR
    Authors: Qingyao Ai, Jingtao Zhan
    Year: N/A
------------------------------------------------------------
 4. Title: Multi-Hierarchical Feature Detection for Large Language Model Generated Text
    Authors: Luyan Zhang, Xinyu Xie
    Year: N/A
------------------------------------------------------------
 5. Title: mdok of KInIT: Robustly Fine-tuned LLM for Binary and Multiclass AI-Generated Text Detection
    Authors: Dom

In [67]:
print(papers[4])

{'title': 'mdok of KInIT: Robustly Fine-tuned LLM for Binary and Multiclass AI-Generated Text Detection', 'authors': ['Dominik Macko'], 'abstract': 'The large language models (LLMs) are able to generate high-quality texts in multiple languages. Such texts are often not recognizable by humans as generated, and therefore present a potential of LLMs for misuse (e.g., plagiarism, spams, disinformation spreading). An automated detection is able to assist humans to indicate the machine-generated texts; however, its robustness to out-of-distribution data is still challenging. This notebook describes our mdok approach in robust detection, based on fine-tuning smaller LLMs for text classification. It is applied to both subtasks of Voight-Kampff Generative AI Detection 2025, providing remarkable performance (1st rank) in both, the binary detection as well as the multiclass classification of various cases of human-AI collaboration.', 'pdf_url': '2025-06-02T14:07:32Z', 'source': 'arxiv'}


In [63]:
# sementic_scholar testing
r2 = search_semantic_scholar("AI-based tuberculosis detection from chest X-ray images", max_results=2)
print(r2)

[]
